In [22]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Set, List, Tuple

### Data Files Definition For Easier Loading

In [7]:
DATA_DIR = Path("data")

TRAIN = DATA_DIR / "train.csv"
VAL  = DATA_DIR / "validation.csv"
TEST = DATA_DIR / "test.csv"
EVALUATION = DATA_DIR / "evaluation.csv"
TRAIN_TRIPLETS = DATA_DIR / "train_triplets.csv"
VAL_TRIPLETS = DATA_DIR / "validation_triplets.csv"
CATALOG = DATA_DIR / "products.csv"
PRODUCT_FEATURES = DATA_DIR / "product_features.npz"

### Loading the Products, ie the catalog

In [8]:
def load_products(path: Path) -> pd.DataFrame:
    product = pd.read_csv(path)
    assert "product_id" in product.columns, "products.csv must have a 'product_id' column"
    assert "catalog" in product.columns, "products.csv must have a 'catalog' column"

    product = product.sort_values("product_id").reset_index(drop=True)
    print(f"[catalog] {len(product)} products loaded")
    print(product["catalog"].value_counts().to_string())
    
    return product

Since the data explorer shows IDs are already zero-based integers,the maps are functions over the valid ranges.
We still build explicit sets/dicts so downstream code is safe if the actual CSVs use arbitrary IDs.

In [9]:
def build_id_maps(product: pd.DataFrame):
    all_item_ids = set(product["product_id"])
    ordinary_item_ids = set(product.loc[product["catalog"] == "ordinary", "product_id"])
    cold_val_ids = set(product.loc[product["catalog"] == "cold_validation", "product_id"])
    cold_test_ids = set(product.loc[product["catalog"] == "cold_test", "product_id"])

    # Map product_id -> integer index (row in features / factor matrices)
    item_id_to_idx = {pid: idx for idx, pid in enumerate(product["product_id"])}

    print(f"\n[id maps] ordinary={len(ordinary_item_ids)}, "
          f"cold_val={len(cold_val_ids)}, cold_test={len(cold_test_ids)}, "
          f"total={len(all_item_ids)}")

    return ordinary_item_ids, all_item_ids, cold_val_ids, cold_test_ids, item_id_to_idx

### Loading the Validation and Test, ie the reviews

In [10]:
def load_val_test(path: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    expected = {"user_id", "product_id", "rating", "timestamp"}
    assert expected.issubset(df.columns), \
        f"{path.name} missing columns: {expected - set(df.columns)}"
    print(f"\n[{split_name}] raw rows: {len(df)}")
    return df

### Safe Filtering for Products

In [11]:
def safe_filter(
    df: pd.DataFrame,
    all_item_ids: set,
    split_name: str
) -> pd.DataFrame:
    
    #Drop any row whose product_id is not in the products.
    before = len(df)
    df = df[df["product_id"].isin(all_item_ids)].reset_index(drop=True)
    dropped = before - len(df)
    if dropped:
        print(f"[{split_name}] dropped {dropped} rows with unknown product_ids")
    else:
        print(f"[{split_name}] all product_ids are in catalog")
    return df

In [12]:
def apply_id_maps(
    df: pd.DataFrame,
    item_id_to_idx: dict,
    split_name: str
) -> pd.DataFrame:
    """
    Add 'item_idx' column: the integer index into factor matrices / feature arrays.
    user_id is already 0-based so we keep it as-is (rename for clarity).
    Rows with unmapped IDs (users or items not seen in catalog) are dropped.
    """
    df = df.copy()

    # Map product_id → item_idx
    df["item_idx"] = df["product_id"].map(item_id_to_idx)

    # user_id is already a clean integer index
    df = df.rename(columns={"user_id": "user_idx"})

    # Drop any NaN - another safety measure
    before = len(df)
    df = df.dropna(subset=["user_idx", "item_idx"]).reset_index(drop=True)
    df["user_idx"] = df["user_idx"].astype(int)
    df["item_idx"] = df["item_idx"].astype(int)
    dropped = before - len(df)
    if dropped:
        print(f"[{split_name}] dropped {dropped} rows after ID mapping")

    print(f"[{split_name}] final rows: {len(df)}, "
          f"unique users: {df['user_idx'].nunique()}, "
          f"unique items: {df['item_idx'].nunique()}")
    return df

### Loading Triplets Files

In [13]:
def load_triplets(path: Path, split_name: str, all_item_ids: set) -> pd.DataFrame:
    df = pd.read_csv(path)
    expected = {"user_id", "positive_id", "comparison_id"}
    assert expected.issubset(df.columns), \
        f"{path.name} missing columns: {expected - set(df.columns)}"

    before = len(df)
    mask = df["positive_id"].isin(all_item_ids) & df["comparison_id"].isin(all_item_ids)
    df = df[mask].reset_index(drop=True)
    dropped = before - len(df)
    if dropped:
        print(f"[{split_name} triplets] dropped {dropped} triplets with unknown IDs")
    else:
        print(f"[{split_name} triplets] all IDs in catalog ({len(df)} triplets)")
    return df

### Loading the feature file

In [19]:
def load_product_features(path: Path) -> np.ndarray:
    #.npz files are archives; we try the key 'features' first, then fall back to the first key found in the archive.
    archive = np.load(path)
    if "Z" in archive:
        Z = archive["Z"]
    elif "features" in archive:
        Z = archive["features"]
    else:
        key = list(archive.keys())[0]
        print(f"[text features] unexpected key '{key}' — using it")
        Z = archive[key]

    assert Z.ndim == 2 and Z.shape[1] == 384, \
        f"Expected shape (num_products, 384), got {Z.shape}"
    print(f"\n[text features] shape: {Z.shape}")
    # check: vectors should be (approximately) unit length
    norms = np.linalg.norm(Z, axis=1)
    print(f"[text features] norm min={norms.min():.4f}, max={norms.max():.4f} "
          f"(should be ≈ 1.0)")
    return Z

### User Positives

In [15]:
def build_user_positives(reviews: pd.DataFrame) -> dict:
    """
    Returns {user_idx: set of item_idx} for all positive interactions
    (rating >= 4) in the given split.
    Used by the Recall@10 / NDCG@10 evaluator.
    """
    pos = reviews[reviews["rating"] >= 4]
    user_positives = (
        pos.groupby("user_idx")["item_idx"]
           .apply(set)
           .to_dict()
    )
    return user_positives

### Main Run for Loading

In [20]:
def load_all_data(data_dir: Path = DATA_DIR) -> dict:
    print("PHASE 1 — DATA LOADING")

    # Products
    products = load_products(data_dir / "products.csv")
    (ordinary_ids,
     all_ids,
     cold_val_ids,
     cold_test_ids,
     item_id_to_idx) = build_id_maps(products)

    # Reviews
    val_reviews  = load_val_test(data_dir / "validation.csv",  "validation")
    test_reviews = load_val_test(data_dir / "test.csv",        "test")

    # evaluation.csv — load and peek at columns so we know what it contains
    # evaluation.csv — ground-truth relevance file for Recall@10 / NDCG@10
    # Columns: task, split, user_id, relevant_product_id
    if (data_dir / "evaluation.csv").exists():
        eval_df = pd.read_csv(data_dir / "evaluation.csv")
        print(f"\n[evaluation.csv] {len(eval_df)} rows, columns: {list(eval_df.columns)}")
        print(eval_df.groupby(["task", "split"]).size().to_string())
    else:
        eval_df = None
        print("\n[evaluation.csv] not found — skipping")

    # Catalog-safe filter
    val_reviews  = safe_filter(val_reviews,  all_ids, "validation")
    test_reviews = safe_filter(test_reviews, all_ids, "test")

    # Apply ID maps
    val_reviews  = apply_id_maps(val_reviews,  item_id_to_idx, "validation")
    test_reviews = apply_id_maps(test_reviews, item_id_to_idx, "test")

    # Triplets (validation only — no test_triplets.csv in the data folder)
    val_triplets = load_triplets(data_dir / "validation_triplets.csv",
                                 "validation", all_ids)

    # Text features
    Z = load_product_features(data_dir / "product_features.npz")

    # Per-user positive sets (for metric computation)
    val_user_positives  = build_user_positives(val_reviews)
    test_user_positives = build_user_positives(test_reviews)

    # Separate ordinary vs cold reviews (needed for cold-start evaluation later)
    val_ordinary_reviews  = val_reviews[val_reviews["item_idx"].map(
        lambda i: products.iloc[i]["catalog"] == "ordinary")]
    val_cold_reviews      = val_reviews[val_reviews["item_idx"].map(
        lambda i: products.iloc[i]["catalog"] == "cold_validation")]
    test_ordinary_reviews = test_reviews[test_reviews["item_idx"].map(
        lambda i: products.iloc[i]["catalog"] == "ordinary")]
    test_cold_reviews     = test_reviews[test_reviews["item_idx"].map(
        lambda i: products.iloc[i]["catalog"] == "cold_test")]

    print("SUMMARY")
    print(f"Users : 170")
    print(f"Ordinary items: {len(ordinary_ids)}")
    print(f"Cold-val items: {len(cold_val_ids)}")
    print(f"Cold-test items: {len(cold_test_ids)}")
    print(f"Val reviews: {len(val_reviews)}")
    print(f"ordinary: {len(val_ordinary_reviews)}")
    print(f"cold: {len(val_cold_reviews)}")
    print(f"Test reviews: {len(test_reviews)}")
    print(f"ordinary: {len(test_ordinary_reviews)}")
    print(f"cold: {len(test_cold_reviews)}")
    print(f"Val triplets: {len(val_triplets)}")
    print(f"Text feature shape : {Z.shape}")

    return {
        # catalog info
        "products":products,
        "eval_df":eval_df,          # raw evaluation.csv (inspect columns first)
        "ordinary_ids":ordinary_ids,
        "all_ids":all_ids,
        "cold_val_ids":cold_val_ids,
        "cold_test_ids":cold_test_ids,
        "item_id_to_idx":item_id_to_idx,
        "num_users":170,
        "num_items":len(products),

        # reviews
        "val_reviews":val_reviews,
        "test_reviews":test_reviews,
        "val_ordinary_reviews":val_ordinary_reviews,
        "val_cold_reviews":val_cold_reviews,
        "test_ordinary_reviews":test_ordinary_reviews,
        "test_cold_reviews":test_cold_reviews,

        # triplets
        "val_triplets":val_triplets,

        # per-user positives
        "val_user_positives":val_user_positives,
        "test_user_positives": test_user_positives,

        # text features
        "Z":Z,
    }

In [21]:
data = load_all_data()
print("\n── Validation reviews sample ──")
print(data["val_reviews"].head())

print("\n── Test reviews sample ──")
print(data["test_reviews"].head())

print("\n── Val triplets sample ──")
print(data["val_triplets"].head())

print("\n── Users with ≥1 positive in validation ──")
print(f"  {len(data['val_user_positives'])} users")

PHASE 1 — DATA LOADING
[catalog] 286 products loaded
catalog
ordinary           206
cold_validation     40
cold_test           40

[id maps] ordinary=206, cold_val=40, cold_test=40, total=286

[validation] raw rows: 358

[test] raw rows: 362

[evaluation.csv] 665 rows, columns: ['task', 'split', 'user_id', 'relevant_product_id']
task      split     
cold      test          217
          validation    217
ordinary  test          110
          validation    121
[validation] all product_ids are in catalog
[test] all product_ids are in catalog
[validation] final rows: 358, unique users: 160, unique items: 122
[test] final rows: 362, unique users: 158, unique items: 119
[validation triplets] all IDs in catalog (121 triplets)

[text features] shape: (286, 384)
[text features] norm min=1.0000, max=1.0000 (should be ≈ 1.0)
SUMMARY
Users : 170
Ordinary items: 206
Cold-val items: 40
Cold-test items: 40
Val reviews: 358
ordinary: 129
cold: 229
Test reviews: 362
ordinary: 120
cold: 242
Val triplet

### Part 2

In [23]:
def build_ground_truth(eval_df: pd.DataFrame) -> Dict[Tuple[str, str, int], Set[int]]:
    """
    Returns a dict keyed by (task, split, user_id) → set of relevant product_ids.

    Example:
        gt[('ordinary', 'validation', 0)] = {128}
        gt[('cold',     'validation', 0)] = {210, 215, 231}
    """
    gt = {}
    for (task, split, user_id), group in eval_df.groupby(["task", "split", "user_id"]):
        gt[(task, split, int(user_id))] = set(group["relevant_product_id"].astype(int))
    return gt


### Rank Candidate

In [24]:
def rank_candidates(
    candidate_ids: np.ndarray,   # shape (num_candidates,)  — product IDs
    scores: np.ndarray,          # shape (num_candidates,)  — higher = more relevant
) -> np.ndarray:
    """
    Returns candidate_ids sorted by (score DESC, product_id ASC).
    """
    # np.lexsort sorts by last key first, so we pass (product_id ASC, -score ASC)
    # which is equivalent to (score DESC, product_id ASC)
    order = np.lexsort((candidate_ids, -scores))
    return candidate_ids[order]

### Recall@k

In [25]:
K = 10

def recall_at_k(ranked_ids: np.ndarray, relevant: Set[int], k: int = K) -> float:
    top_k = set(ranked_ids[:k])
    hits  = len(top_k & relevant)
    return hits / len(relevant) if relevant else 0.0

### NDGC@k

In [26]:
def ndcg_at_k(ranked_ids: np.ndarray, relevant: Set[int], k: int = K) -> float:
    if not relevant:
        return 0.0

    # DCG: score the actual ranking
    dcg = 0.0
    for pos, pid in enumerate(ranked_ids[:k], start=1):  # pos is 1-based
        if pid in relevant:
            dcg += 1.0 / np.log2(pos + 1)

    # IDCG: score the ideal ranking (relevant items placed at positions 1, 2, ...)
    num_relevant_in_k = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(pos + 1) for pos in range(1, num_relevant_in_k + 1))

    return dcg / idcg if idcg > 0 else 0.0

### Scoring all candidates for each user

In [33]:
def score_candidates(
    user_idx: int,
    candidate_ids: np.ndarray,   # product IDs (not indices) to score
    model: dict,
    item_id_to_idx: dict,
    task: str,                   # 'ordinary' or 'cold'
) -> np.ndarray:
    """
    Returns a score array aligned with candidate_ids.

    model must contain:
        P  : (num_users, k)        user factor matrix
        Q  : (num_ordinary, k)     item factor matrix  (ordinary only)
        W  : (k, 384)              text projection matrix
        b  : (num_ordinary,)       item biases         (ordinary only)
        Z  : (num_products, 384)   text features       (all products)
    """
    P = model["P"]
    Q = model["Q"]
    W = model["W"]
    b = model["b"]
    Z = model["Z"]

    p_u = P[user_idx]  # shape (k,)

    scores = np.empty(len(candidate_ids), dtype=np.float64)

    for j, pid in enumerate(candidate_ids):
        idx = item_id_to_idx[pid]
        z_i = Z[idx]            # text feature vector, shape (384,)
        Wz  = W @ z_i           # projected text, shape (k,)
        ordinary_idx = model["ordinary_idx"]

        if task == "ordinary":
           q_idx = ordinary_idx[pid]
           q_i   = Q[q_idx]
           bi    = b[q_idx]      # learned item bias
           scores[j] = bi + p_u @ (q_i + Wz)
        else:
            # cold: q_i = 0, b_i = 0
            scores[j] = p_u @ Wz

    return scores

### Main Evaluator

In [34]:
def evaluate(
    model: dict,
    eval_df: pd.DataFrame,
    item_id_to_idx: dict,
    ordinary_ids: Set[int],
    cold_val_ids: Set[int],
    cold_test_ids: Set[int],
    task: str,    # 'ordinary' or 'cold'
    split: str,   # 'validation' or 'test'
    k: int = K,
    verbose: bool = True,
) -> dict:
    """
    Evaluate a model on one (task, split) combination.
    Returns dict with keys: recall, ndcg, per_user_recall, per_user_ndcg.
    """
    # Ground truth for this task/split
    gt = build_ground_truth(eval_df)

    # Candidate set depends on task + split
    if task == "ordinary":
        candidate_ids = np.array(sorted(ordinary_ids))
    elif task == "cold" and split == "validation":
        candidate_ids = np.array(sorted(cold_val_ids))
    elif task == "cold" and split == "test":
        candidate_ids = np.array(sorted(cold_test_ids))
    else:
        raise ValueError(f"Unknown task/split combination: {task}/{split}")

    # Users to evaluate = those who appear in evaluation.csv for this task+split
    users = [uid for (t, s, uid) in gt if t == task and s == split]

    recalls, ndcgs = [], []

    for user_idx in users:
        relevant = gt[(task, split, user_idx)]

        # Score every candidate
        scores = score_candidates(
            user_idx, candidate_ids, model, item_id_to_idx, task
        )

        # Rank with tie-breaking
        ranked = rank_candidates(candidate_ids, scores)

        # Compute metrics
        recalls.append(recall_at_k(ranked, relevant, k))
        ndcgs.append(ndcg_at_k(ranked, relevant, k))

    mean_recall = float(np.mean(recalls))
    mean_ndcg   = float(np.mean(ndcgs))

    if verbose:
        print(f"[{task:8s} | {split:10s}]  "
              f"Recall@{k}={mean_recall:.4f}  NDCG@{k}={mean_ndcg:.4f}  "
              f"({len(users)} users)")

    return {
        "recall":          mean_recall,
        "ndcg":            mean_ndcg,
        "per_user_recall": recalls,
        "per_user_ndcg":   ndcgs,
        "num_users":       len(users),
    }

### Evaluating the four combinations

In [35]:
def evaluate_all(
    model: dict,
    eval_df: pd.DataFrame,
    item_id_to_idx: dict,
    ordinary_ids: Set[int],
    cold_val_ids: Set[int],
    cold_test_ids: Set[int],
    k: int = K,
) -> dict:
    """
    Runs evaluate() for all four combinations and returns a nested dict:
        results[task][split] = {recall, ndcg, ...}
    """
    print(f"\n{'='*60}")
    print(f"EVALUATION  (K={k})")
    print(f"{'='*60}")

    results = {}
    for task in ["ordinary", "cold"]:
        results[task] = {}
        for split in ["validation", "test"]:
            results[task][split] = evaluate(
                model, eval_df, item_id_to_idx,
                ordinary_ids, cold_val_ids, cold_test_ids,
                task=task, split=split, k=k,
            )
    return results

### Running a test

In [30]:
def _run_unit_tests():
    print("\n── Unit tests ──")

    # Test 1: relevant item IS in top-10 → Recall=1.0, NDCG > 0
    ids    = np.arange(20)
    scores = np.arange(20, dtype=float)   # item 19 scores highest
    ranked = rank_candidates(ids, scores)
    assert ranked[0] == 19, "highest score should rank first"
    r = recall_at_k(ranked, {19}, k=10)
    n = ndcg_at_k(ranked,   {19}, k=10)
    assert r == 1.0,  f"expected Recall=1.0, got {r}"
    assert n == 1.0,  f"expected NDCG=1.0 (item at pos 1), got {n}"
    print("  Test 1 passed: relevant item at rank 1 → Recall=1.0, NDCG=1.0")

    # Test 2: relevant item NOT in top-10 → both metrics = 0
    r = recall_at_k(ranked, {0}, k=10)   # item 0 is last
    n = ndcg_at_k(ranked,   {0}, k=10)
    assert r == 0.0, f"expected Recall=0.0, got {r}"
    assert n == 0.0, f"expected NDCG=0.0, got {n}"
    print("  Test 2 passed: relevant item outside top-10 → Recall=0.0, NDCG=0.0")

    # Test 3: tie-breaking by product_id ASC
    ids2    = np.array([5, 3, 7])
    scores2 = np.array([1.0, 1.0, 1.0])   # all equal scores
    ranked2 = rank_candidates(ids2, scores2)
    assert list(ranked2) == [3, 5, 7], f"tie-break failed: {ranked2}"
    print("  Test 3 passed: equal scores → sorted by product_id ascending")

    # Test 4: NDCG penalises lower rank
    # relevant item at rank 1 vs rank 2 — rank 1 must give higher NDCG
    ids3 = np.array([10, 20, 30])
    # Case A: relevant item (10) at rank 1
    scoresA = np.array([3.0, 2.0, 1.0])
    ndcgA   = ndcg_at_k(rank_candidates(ids3, scoresA), {10})
    # Case B: relevant item (10) at rank 2
    scoresB = np.array([2.0, 3.0, 1.0])
    ndcgB   = ndcg_at_k(rank_candidates(ids3, scoresB), {10})
    assert ndcgA > ndcgB, "rank-1 should give higher NDCG than rank-2"
    print(f"  Test 4 passed: rank-1 NDCG={ndcgA:.4f} > rank-2 NDCG={ndcgB:.4f}")

    # Test 5: multiple relevant items
    ids4   = np.arange(10)
    scores4 = np.arange(10, dtype=float)
    ranked4 = rank_candidates(ids4, scores4)  # ranked: 9,8,7,6,5,4,3,2,1,0
    r = recall_at_k(ranked4, {9, 8, 0}, k=10)  # 9 and 8 in top-2, 0 at rank 10
    assert r == 1.0, f"all 3 relevant in top-10 → Recall=1.0, got {r}"
    print("  Test 5 passed: multiple relevant items all in top-10 → Recall=1.0")

    print("  All unit tests passed ✓\n")

_run_unit_tests()



── Unit tests ──
  Test 1 passed: relevant item at rank 1 → Recall=1.0, NDCG=1.0
  Test 2 passed: relevant item outside top-10 → Recall=0.0, NDCG=0.0
  Test 3 passed: equal scores → sorted by product_id ascending
  Test 4 passed: rank-1 NDCG=1.0000 > rank-2 NDCG=0.6309
  Test 5 passed: multiple relevant items all in top-10 → Recall=1.0
  All unit tests passed ✓



In [36]:
rng = np.random.default_rng(seed=616)

# Build the ordinary_idx map — product_id → row in Q and b
ordinary_ids_sorted = sorted(data["ordinary_ids"])
ordinary_idx = {pid: i for i, pid in enumerate(ordinary_ids_sorted)}

dummy_model = {
    "P": rng.standard_normal((170, 32)),
    "Q": rng.standard_normal((206, 32)),
    "W": rng.standard_normal((32, 384)),
    "b": rng.standard_normal((206,)),
    "Z": data["Z"],
    "ordinary_idx": ordinary_idx,   # ← new
}

results = evaluate_all(
    model          = dummy_model,
    eval_df        = data["eval_df"],
    item_id_to_idx = data["item_id_to_idx"],
    ordinary_ids   = data["ordinary_ids"],
    cold_val_ids   = data["cold_val_ids"],
    cold_test_ids  = data["cold_test_ids"],
)


EVALUATION  (K=10)
[ordinary | validation]  Recall@10=0.0496  NDCG@10=0.0217  (121 users)
[ordinary | test      ]  Recall@10=0.0273  NDCG@10=0.0105  (110 users)
[cold     | validation]  Recall@10=0.2084  NDCG@10=0.1058  (119 users)
[cold     | test      ]  Recall@10=0.2349  NDCG@10=0.1279  (114 users)
